# Strategy 6. Adapters

In [1]:
try:
    del base_model
except:
    pass
try:
    del adapter_model
except:
    pass
try:
    del tokenizer
except:
    pass

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Define paths
model_path = "/Users/nunocalaim/Documents/Mistral-7B-Instruct-v0.1"  # Replace with the correct model name on Hugging Face
adapter_path = "/Users/nunocalaim/Documents/mistral_instruct_cypher"  # Replace with the adapter path or Hugging Face repo link

# Load the Mistral base model
base_model = AutoModelForCausalLM.from_pretrained(model_path)

# Apply the adapter
adapter_model = PeftModel.from_pretrained(base_model, "/Users/nunocalaim/Documents/mistral_instruct_cypher")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



Wrappers for LLMs and KG

In [6]:
chat_models = {
    'mistral': base_model,
    'mistral_adaptors': adapter_model,
}

In [7]:
# Cypher data extraction
import re
def extract_cypher(message):
    text = message
    try:
        pattern = r"```cypher(.*?)```"
        matches = re.findall(pattern, text, re.DOTALL)
        if len(matches) > 0:
            return [match.strip() for match in matches]
        else:
            pattern = r"```(.*?)```"
            matches = re.findall(pattern, text, re.DOTALL)
            return [match.strip() for match in matches]
            
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [8]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [9]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [10]:
questions = {
    "tdp-als": "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)?",
    "tdp-cancer": "What is the evidence linking TDP-43 to cancer in animal models?",
    "braf-melanoma": "What (or is there) is the clinical evidence linking BRAF to Melanoma?",
}

In [11]:
from langchain.schema import HumanMessage, SystemMessage

niter = 10
# niter = 2

todo = [(m, q, i) for q in questions.items() for m in chat_models for i in range(niter)]

In [ ]:
def send_llm(model, system_prompt, user_prompt):
    inputs = tokenizer(system_prompt + "\n" + user_prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import json

# Load the JSON file
with open("normal_schema.json", "r") as file:
    json_data = json.load(file)

# Convert the JSON object to a string
json_string = json.dumps(json_data)

# Create a string that includes the JSON string as a substring
result_string = f"This is my string with JSON data: {json_string}"

# Print the result
print(result_string)


In [ ]:
system_prompt = f"""
Generate a database query in Cypher that answers the user's question.

This is the schema of the graph database
{result_string}

Generate a Cypher query.
Only return the query encapsulated in triple backticks with cypher indicating it is a cypher query,
without any additional text, symbols or characters --- just the query statement.

IMPORTNAT:
- Always escape labels containing dots and other not allowed symbols with backticks!
- Make queries case-insensitive.
"""

In [ ]:
def run_llm_6(model, question):
    try:

        final_query_response = send_llm(model, system_prompt, question)
        print(f"Got the following query: {final_query_response}")

        return final_query_response  # Return the final Cypher query

    except Exception as e:
        print(e)
        return None


In [ ]:
llm_answers = []
for llm_model, question, iter in tqdm(todo, desc="Prompting LLM"):
    file_p = f'../../data/{llm_model}_{iter}_{question[0]}.txt'
    if os.path.exists(file_p):
        with open(file_p, "r") as f:
            llm_answers.append(f.read())
    else:
        print(f"Model: {llm_model}, Prompt: {question[1]}, iter {iter}")
        answer = run_llm_6(chat_models[llm_model], question[1])
        if answer:
            with open(file_p, "w") as f:
                f.write(answer)
        llm_answers.append(answer)

In [ ]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

In [ ]:
results = process_results(todo, llm_answers, cypher_results)
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Function to clean illegal characters
def clean_illegal_characters(value):
    if isinstance(value, str):
        # Remove control characters
        value = re.sub(r"[\x00-\x09\x0B-\x1F\x7F-\x9F]", "", value)
        # Replace multiline text with a single-line equivalent
        # value = value.replace("\n", " ").replace("\r", " ")
    return value

# Apply cleaning to the entire DataFrame
results_df = results_df.apply(lambda x: x.map(clean_illegal_characters))

In [ ]:
results_df.to_excel("06-evaluations.xlsx", index=False)


In [ ]:
with open("06-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df